# 09 — External Triggers & Synchronization

This notebook covers every mechanism available for coordinating Python and IviumSoft:

- **Status parameter** — shared integer that both Python and IviumSoft Batch scripts can read and write
- **Digital output trigger** — Python drives a hardware line; IviumSoft Batch halts until it sees the edge
- **Analog output trigger** — Python raises a DAC voltage above a threshold; IviumSoft Batch halts until the level is met
- **IviumSoft → Python** — IviumSoft Batch launches a Python script via `ExecuteProgram` and waits for it to exit
- **Multi-instance broadcast** — send a trigger to all active IviumSoft instances simultaneously

### When to use each mechanism

| Mechanism | Best for |
|-----------|----------|
| Status parameter | Pure software coordination — no extra wiring, works over any connection |
| Digital trigger | Hard real-time edge: sub-ms latency, works with external instruments too |
| Analog trigger | Threshold-based start: voltage level from a sensor or DAC |
| ExecuteProgram | IviumSoft drives Python (post-processing, notifications, file generation) |

### Prerequisites
- IviumSoft installed and running
- Driver open (see `01_getting_started.ipynb`)
- For digital/analog triggers: IviumSoft Batch method configured with the matching `DirectCommand`
- Error handling reference: `01_getting_started.ipynb` — Section 4

In [ ]:
import time
from pyvium import Pyvium

Pyvium.open_driver()
print(f"Active instances: {Pyvium.get_active_iviumsoft_instances()}")

## 1. The Status Parameter

`set_status_par(value)` / `get_status_par()` expose a global integer shared between Python and IviumSoft. It is **per-instance** — each IviumSoft window maintains its own independent value.

**Key facts from the IviumSoft manual:**
- Initialized to **`-1`** at PC startup
- **Retains its last-set value** until the PC is restarted — it does not reset between measurements or driver open/close cycles
- Both Python (via pyvium) and IviumSoft Batch scripts can read and write it
- The DLL function is `IV_setstatuspar` / `IV_getstatuspar`

### Recommended convention

| Value | Who sets it | Meaning |
|-------|------------|---------|
| `-1` | PC startup default | Uninitialized |
| `0` | Python | Reset / idle |
| `1` | Python | "Ready — IviumSoft may start" |
| `2` | IviumSoft Batch | "Measurement running" |
| `3` | IviumSoft Batch | "Measurement done" |
| `-99` | Either | Abort / error |

> **Always reset to `0` explicitly** at the start of each workflow. Because the value persists until PC restart, a leftover value from a previous session can cause the IviumSoft Batch `WaitForStatusPar` to fire immediately and skip the intended wait.

In [ ]:
Pyvium.select_iviumsoft_instance(1)

# Always reset before a new workflow
Pyvium.set_status_par(0)
print(f"Reset to 0")

# Read it back
val = Pyvium.get_status_par()
print(f"Current value: {val}")

## 2. Python → IviumSoft: Signal IviumSoft to Start

Pattern: Python does its preparation work, then sets the status parameter to `1`. An IviumSoft Batch script that is already running pauses at a `WaitForStatusPar` line until it sees `1`, then proceeds.

**IviumSoft Batch configuration:**
```
DirectCommand:
  WaitForStatusPar  ✓   Value = 1   Timeout = 300 s
```

**Flow:**
```
Python:    set_status_par(0)        ← reset
Python:    ... prepare experiment ...
Python:    set_status_par(1)        ← fire trigger
IviumSoft: WaitForStatusPar exits   ← Batch continues
```

In [ ]:
Pyvium.select_iviumsoft_instance(1)
Pyvium.set_status_par(0)   # reset
print("Status par reset to 0 (idle)")

# ... your preparation work here ...
time.sleep(0.5)            # simulates preparation time

Pyvium.set_status_par(1)   # fire the trigger
print("Triggered IviumSoft (status par = 1)")

## 3. IviumSoft → Python: Wait for IviumSoft to Finish

Pattern: Python polls `get_status_par()` in a loop, waiting for IviumSoft to set a specific value when it finishes. The IviumSoft Batch side uses a `SetStatusPar` DirectCommand to write the value.

**IviumSoft Batch configuration:**
```
DirectCommand:
  SetStatusPar  ✓   Value = 3     ← written when measurement finishes
```

**Flow:**
```
IviumSoft: ... runs experiment ...
IviumSoft: SetStatusPar = 3        ← signals Python
Python:    get_status_par() → 3    ← poll loop exits
Python:    collect_data()
```

In [ ]:
def wait_for_status(instance: int, target: int, timeout_s: float = 120.0) -> bool:
    """Block until the given instance's status par reaches target, or timeout."""
    Pyvium.select_iviumsoft_instance(instance)
    t0 = time.time()
    while True:
        current = Pyvium.get_status_par()
        if current == target:
            return True
        if time.time() - t0 > timeout_s:
            print(f"Timeout: instance {instance} never reached status={target}")
            return False
        time.sleep(0.2)

print("wait_for_status() defined")

# Wait up to 60 s for IviumSoft to set status par to 3 ("done")
# done = wait_for_status(instance=1, target=3, timeout_s=60.0)
# if done:
#     print("IviumSoft finished — collecting data")

## 4. The Full Bidirectional Contract

Combining sections 2 and 3 produces a complete handshake: Python signals IviumSoft to start, IviumSoft signals Python when done.

| Step | Python side | IviumSoft Batch side |
|------|-------------|----------------------|
| 1 | `set_status_par(0)` — reset | — |
| 2 | (prepare) | `WaitForStatusPar` (value=1, timeout=300s) — waits |
| 3 | `set_status_par(1)` — go | `WaitForStatusPar` exits, Batch proceeds |
| 4 | `wait_for_status(1, target=3)` — polls | (runs experiment) → `SetStatusPar` (value=3) |
| 5 | Poll exits, collect data | — |

> **Timeout on both sides:** Always configure a timeout in `WaitForStatusPar` (IviumSoft side) and in `wait_for_status()` (Python side). A missing signal on either side must never permanently stall the sequence.

In [ ]:
def run_with_handshake(instance: int, timeout_s: float = 120.0):
    Pyvium.select_iviumsoft_instance(instance)

    # Step 1: reset
    Pyvium.set_status_par(0)
    print(f"[inst {instance}] Reset — status=0")

    # Step 2-3: signal IviumSoft to start
    Pyvium.set_status_par(1)
    print(f"[inst {instance}] Triggered — status=1")

    # Step 4: wait for IviumSoft to report done (status=3)
    done = wait_for_status(instance, target=3, timeout_s=timeout_s)
    if done:
        print(f"[inst {instance}] IviumSoft done — status=3")
    return done

print("run_with_handshake() defined")
# run_with_handshake(instance=1, timeout_s=60.0)

## 5. Digital Output Trigger

Python drives a digital output line wired to the CompactStat's external digital input. IviumSoft Batch halts on `WaitForDigIn1` until it detects the edge.

**IviumSoft Batch configuration:**
```
DirectCommand:
  WaitForDigIn1  ✓   WaitForHi = ✓   Timeout = 60 s
```

**Hardware setup:**
```
CompactStat external port
  Digital Output 0  ──────────────────────  Digital Input 1
                           (short wire)
```

`set_digital_output(bitmask)` controls all digital output lines simultaneously. Each bit corresponds to one line:
```
0b00000001  →  line 0 HIGH
0b00000100  →  line 2 HIGH
0b00000000  →  all lines LOW
```

> **Timing:** IviumSoft polls `DigIn1` roughly every 10–50 ms. A pulse shorter than ~50 ms may be missed. Hold the line HIGH for at least 100 ms before releasing it.

In [ ]:
TRIGGER_LINE = 0   # digital output line wired to device's DigIn1
HOLD_TIME_S  = 0.1 # hold HIGH for 100 ms — enough for IviumSoft to detect

def trigger_digital(line: int = TRIGGER_LINE, hold_s: float = HOLD_TIME_S):
    """Assert a digital output line HIGH, hold, then release."""
    Pyvium.set_digital_output(1 << line)
    print(f"Digital line {line} → HIGH")
    time.sleep(hold_s)
    Pyvium.set_digital_output(0)
    print(f"Digital line {line} → LOW (released)")

def read_digital_inputs() -> dict:
    """Return the current state of all digital input lines as a dict."""
    bitmask = Pyvium.get_digital_input()
    return {line: bool(bitmask & (1 << line)) for line in range(8)}

print("Digital trigger helpers defined")

# Read the current digital input state
inputs = read_digital_inputs()
for line, state in inputs.items():
    print(f"  DigIn {line}: {'HIGH' if state else 'LOW'}")

# Fire the trigger
# trigger_digital()

## 6. Analog Output Trigger

Python raises DAC channel 0 above a threshold voltage. IviumSoft Batch halts at `WaitForAn1` until Analog Input 1 exceeds the configured threshold.

**IviumSoft Batch configuration:**
```
DirectCommand:
  WaitForAn1  ✓   UntilAn1 > 2.5 V   Timeout = 60 s
```

**Hardware setup:**
```
CompactStat external port
  DAC channel 0  ──────────────────────  Analog Input 1
                        (short wire)
```

**Choosing the threshold:**
- Set the IviumSoft threshold to a value that is clearly above your idle DAC voltage and below your trigger voltage
- Example: idle = 0 V, trigger = 3.0 V, IviumSoft threshold = 2.5 V
- Avoid thresholds near the idle level — DAC noise could cause false triggers

> **DAC output range:** typically ±10 V. Check your device specification before exceeding ±5 V.

In [ ]:
TRIGGER_VOLTAGE = 3.0   # V — must exceed the IviumSoft Batch threshold
IDLE_VOLTAGE    = 0.0   # V — safe level when not triggering
DAC_CHANNEL     = 0     # DAC channel wired to Analog Input 1
HOLD_TIME_S     = 0.5   # hold above threshold long enough for IviumSoft to detect

def trigger_analog(
    channel: int = DAC_CHANNEL,
    trigger_v: float = TRIGGER_VOLTAGE,
    idle_v: float = IDLE_VOLTAGE,
    hold_s: float = HOLD_TIME_S,
):
    """Raise DAC to trigger_v, hold, then return to idle_v."""
    Pyvium.set_dac(channel, trigger_v)
    print(f"DAC {channel} → {trigger_v} V (trigger)")
    time.sleep(hold_s)
    Pyvium.set_dac(channel, idle_v)
    print(f"DAC {channel} → {idle_v} V (idle)")

def read_analog_inputs(n_channels: int = 4) -> dict:
    """Read the first n_channels ADC inputs and return as a dict."""
    return {ch: Pyvium.get_adc(ch) for ch in range(n_channels)}

print("Analog trigger helpers defined")

# Read current ADC values
adc = read_analog_inputs()
for ch, v in adc.items():
    print(f"  ADC {ch}: {v:.4f} V")

# Fire the trigger
# trigger_analog()

## 7. IviumSoft Launches Python (`ExecuteProgram`)

The inverse direction: an IviumSoft Batch script launches a Python script at a specific point in its sequence and optionally waits for it to exit before continuing.

**IviumSoft Batch configuration:**
```
DirectCommand:
  ExecuteProgram     ✓
  Program Name:      python.exe
  Path:              C:\Users\...\Scripts\
  Command Line:      C:\path\to\my_script.py --output C:\data\result.csv
  WaitUntilFinished: ✓
```

**Typical sequence:**
```
[Batch line N]    ExecuteMethod           ← run electrochemical experiment
[Batch line N+1]  ExecuteProgram          ← launch Python, wait for exit
[Batch line N+2]  Loop back / next scan   ← Python result already on disk
```

**Use cases:**
- Post-process data immediately after each scan inside a Batch loop
- Send a notification (email, Slack) when a long experiment completes
- Read the measurement result and write a parameter file for the next Batch line
- Run quality control — abort the Batch if the result is out of range

**Exit code convention:**
IviumSoft does not inspect the Python exit code — it only waits for the process to finish. To signal back to IviumSoft from Python, write a file or use `set_status_par()` before exiting.

In [ ]:
# Template for a script launched by IviumSoft via ExecuteProgram
# Save as a standalone .py file; IviumSoft will call it and wait for it to exit.

TEMPLATE = '''
import sys
import argparse
from pyvium import Pyvium

parser = argparse.ArgumentParser()
parser.add_argument("--output", required=True)
args = parser.parse_args()

Pyvium.open_driver()
Pyvium.select_iviumsoft_instance(1)

n = Pyvium.get_available_data_points_number()
rows = [Pyvium.get_data_point(i) for i in range(1, n + 1)]

import csv
with open(args.output, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["E (V)", "I (A)", "aux"])
    writer.writerows(rows)

# Optionally signal back to IviumSoft via status par
Pyvium.set_status_par(10)   # "Python post-processing done"

Pyvium.close_driver()
sys.exit(0)
'''

print("ExecuteProgram script template:")
print(TEMPLATE)

## 8. Multi-Instance Broadcast

To trigger all active IviumSoft instances nearly simultaneously, loop through them and set the status parameter on each. The loop itself takes only a few milliseconds, so all instances receive the signal within one IviumSoft polling cycle.

This is the standard pattern for starting parallel experiments on multiple devices at the same time.

In [ ]:
def broadcast_status(value: int):
    """Set the status parameter on all active IviumSoft instances."""
    instances = Pyvium.get_active_iviumsoft_instances()
    for inst in instances:
        Pyvium.select_iviumsoft_instance(inst)
        Pyvium.set_status_par(value)
    print(f"Broadcast status={value} to instances {instances}")
    return instances

def wait_all_status(instances: list, target: int, timeout_s: float = 120.0) -> bool:
    """Wait until every instance's status par equals target, or timeout."""
    remaining = set(instances)
    t0 = time.time()
    while remaining:
        finished = set()
        for inst in remaining:
            Pyvium.select_iviumsoft_instance(inst)
            if Pyvium.get_status_par() == target:
                finished.add(inst)
                print(f"  Instance {inst}: status={target}")
        remaining -= finished
        if remaining and time.time() - t0 > timeout_s:
            print(f"Timeout — still waiting on instances {remaining}")
            return False
        if remaining:
            time.sleep(0.2)
    return True

print("Broadcast helpers defined")

# Reset all, then trigger all
instances = broadcast_status(0)   # reset
time.sleep(0.5)
broadcast_status(1)               # go

## 9. Choosing Between Trigger Types

### Status parameter vs hardware triggers

| Criterion | Status parameter | Digital trigger | Analog trigger |
|-----------|-----------------|-----------------|----------------|
| Wiring required | None | Yes (1 wire) | Yes (1 wire) |
| Latency | ~200 ms (poll cycle) | ~50 ms (IviumSoft poll) | ~50 ms (IviumSoft poll) |
| Works across network | Yes | No | No |
| Reversible signal | Yes (set again) | Yes (release line) | Yes (lower DAC) |
| IviumSoft Batch side | `WaitForStatusPar` | `WaitForDigIn1` | `WaitForAn1` |
| Can also receive signal | Yes (`get_status_par`) | Yes (`get_digital_input`) | Yes (`get_adc`) |

### Timeout matters on both sides

All three IviumSoft `WaitFor...` DirectCommands accept a **Timeout** setting. If the trigger never arrives, the Batch continues after the timeout expires rather than hanging permanently. Always configure a timeout that is:
- Long enough to cover the worst-case Python preparation time
- Short enough to fail fast if something went wrong

Match the Python-side `timeout_s` in your `wait_for_status()` call to the same time budget.

## 10. Complete Trigger Workflow Template

A self-contained example that:
1. Resets all instances
2. Fires a software trigger to all instances simultaneously
3. Waits for all instances to report done
4. Collects data from all instances

In [ ]:
import time
import pandas as pd
from pyvium import Pyvium

STATUS_IDLE    = 0
STATUS_GO      = 1
STATUS_RUNNING = 2
STATUS_DONE    = 3
STATUS_ABORT   = -99
TRIGGER_TIMEOUT_S = 120.0

Pyvium.open_driver()
instances = Pyvium.get_active_iviumsoft_instances()
print(f"Devices: {instances}")

# Step 1: reset all
broadcast_status(STATUS_IDLE)

# Step 2: connect all
for inst in instances:
    Pyvium.select_iviumsoft_instance(inst)
    Pyvium.connect_device()
    print(f"  Instance {inst}: connected")

# Step 3: trigger all simultaneously
# IviumSoft Batch on each instance must be configured with WaitForStatusPar (value=1)
broadcast_status(STATUS_GO)

# Step 4: wait for all to finish (IviumSoft sets status=3 via SetStatusPar)
all_done = wait_all_status(instances, target=STATUS_DONE, timeout_s=TRIGGER_TIMEOUT_S)

if not all_done:
    broadcast_status(STATUS_ABORT)
    print("Aborted — not all instances finished in time")
else:
    # Step 5: collect data from all
    results = {}
    for inst in instances:
        Pyvium.select_iviumsoft_instance(inst)
        n = Pyvium.get_available_data_points_number()
        rows = [Pyvium.get_data_point(i) for i in range(1, n + 1)]
        results[inst] = pd.DataFrame(rows, columns=["E (V)", "I (A)", "aux"])
        print(f"  Instance {inst}: {len(rows)} points")

Pyvium.close_driver()
print("Done")

## Cleanup

In [ ]:
# Reset all digital and analog outputs to safe state
Pyvium.set_digital_output(0)
Pyvium.set_dac(0, 0.0)
Pyvium.set_dac(1, 0.0)
Pyvium.close_driver()
print("Outputs reset, driver closed")

---

## Summary

### Status parameter

| Task | Python | IviumSoft Batch |
|------|--------|-----------------|
| Reset | `set_status_par(0)` | — |
| Signal IviumSoft to start | `set_status_par(1)` | `WaitForStatusPar` (value=1, timeout) |
| Wait for IviumSoft to finish | `get_status_par()` in poll loop | `SetStatusPar` (value=3) |
| Broadcast to all instances | loop `select_iviumsoft_instance(n)` + `set_status_par(v)` | — |
| Initial value at PC startup | — | **-1** (retains until PC restart) |

### Hardware triggers

| Type | Python sends | IviumSoft Batch waits on | Read-back |
|------|-------------|--------------------------|----------|
| Digital | `set_digital_output(bitmask)` | `WaitForDigIn1` (HI/LO, timeout) | `get_digital_input()` |
| Analog | `set_dac(0, voltage)` | `WaitForAn1 > threshold` (timeout) | `get_adc(channel)` |

### IviumSoft-driven Python

| Method | IviumSoft Batch | Python side |
|--------|----------------|-------------|
| Launch script | `ExecuteProgram` + `WaitUntilFinished` | standalone `.py` file, exits with `sys.exit(0)` |

### Safety checklist

- Always reset status par to `0` before a new workflow
- Always configure a timeout in `WaitForStatusPar` / `WaitForDigIn1` / `WaitForAn1` on the IviumSoft side
- Always set a matching `timeout_s` on the Python poll side
- Reset DAC to `0 V` and digital outputs to `0` in your cleanup cell
- Catch `IllegalCommandError` when calling `abort_method()` defensively

---

## Related notebooks

| Notebook | Topic |
|----------|-------|
| `04_direct_mode_signals.ipynb` | DAC output, ADC input, digital I/O reference |
| `06_method_mode.ipynb` | Running and monitoring methods, `abort_method()` |
| `08_batch_and_synchronization.ipynb` | Multi-device setpoints, parallel measurements, synchronized channel start |